In [1]:
import torch
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from PIL import Image, ImageDraw, ImageFont

model_id = r"E:\Snapfolia - CS\grounding-dino-tiny"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)

image = Image.open(r"F:\SNAP-DS-36\Scramble Egg\env_Scramble Egg A (2).jpg")

text = "a leaf. a leaves."

inputs = processor(images=image, text=text, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)

results = processor.post_process_grounded_object_detection(
    outputs,
    inputs.input_ids,
    box_threshold=0.3,
    text_threshold=0.3,
    target_sizes=[image.size[::-1]]
)

# Draw bounding boxes
draw = ImageDraw.Draw(image)
font = ImageFont.load_default()
output_file = "leaf_coordinates.txt"

# Get image dimensions
image_width, image_height = image.size

# Check if any detections were made
if len(results) > 0:
    # Get the first (and likely only) result
    pred_boxes = results[0]["boxes"]
    pred_labels = results[0]["labels"]
    pred_scores = results[0]["scores"]
    
    with open(output_file, 'w') as f:
        # Draw each detected object
        for i, (box, label, score) in enumerate(zip(pred_boxes, pred_labels, pred_scores), 1):
            # Convert box coordinates to integers
            box = [int(b) for b in box]
            
            # Draw the bounding box with increased thickness (width=5)
            draw.rectangle(box, outline="red", width=5)
            
            # Add label and score
            label_text = f"{label}: {score:.2f}"
            draw.text((box[0], box[1]-10), label_text, fill="red", font=font)
            
            # YOLO format: class_id x_center y_center width height
            x_center = (box[0] + box[2]) / 2 / image_width
            y_center = (box[1] + box[3]) / 2 / image_height
            width = (box[2] - box[0]) / image_width
            height = (box[3] - box[1]) / image_height
            
            # Write coordinates to file in YOLO format
            f.write(f"0 {x_center} {y_center} {width} {height}\n")

            # Print detection details
            print(f"Leaf {i}: Score = {score:.2f}, Box = {box}")

        # Print detection details
        print(f"Detected {len(pred_boxes)} objects:")
        for label, score, box in zip(pred_labels, pred_scores, pred_boxes):
            print(f"- {label}: Score = {score:.2f}, Box = {box}")

        # Save or show the image
        image.save("detected_leaves.jpg")
else:
    print("No objects detected.")

Leaf 1: Score = 0.42, Box = [1796, 633, 2339, 1109]
Leaf 2: Score = 0.44, Box = [1077, 9, 3015, 3005]
Leaf 3: Score = 0.40, Box = [1984, 196, 2576, 646]
Leaf 4: Score = 0.40, Box = [1301, 413, 1789, 862]
Leaf 5: Score = 0.37, Box = [1183, 792, 1698, 1204]
Leaf 6: Score = 0.33, Box = [1105, 1159, 1647, 1498]
Leaf 7: Score = 0.37, Box = [1533, 7, 1970, 459]
Leaf 8: Score = 0.30, Box = [1734, 1052, 2182, 1613]
Leaf 9: Score = 0.31, Box = [1598, 1944, 1859, 2661]
Detected 9 objects:
- a leaf: Score = 0.42, Box = tensor([1796.9335,  633.4043, 2339.9988, 1109.1207], device='cuda:0')
- a leaves: Score = 0.44, Box = tensor([1077.0564,    9.0847, 3015.2966, 3005.9016], device='cuda:0')
- a leaf: Score = 0.40, Box = tensor([1984.0519,  196.8840, 2576.6445,  646.8168], device='cuda:0')
- a leaf: Score = 0.40, Box = tensor([1301.9294,  413.2061, 1789.6742,  862.0345], device='cuda:0')
- a leaf: Score = 0.37, Box = tensor([1183.7864,  792.6502, 1698.0892, 1204.2573], device='cuda:0')
- a leaf: Scor